# Tutorial 4: Finite Differences and Numerical Gradients

**Prerequisites:** Tutorial 1 — What Is an Optimization Problem?, Tutorial 3 — Gradient Descent from Scratch  
**Up Next:** Tutorial 5 — Line Search Methods

---

## Concept

In Tutorial 3 we used a forward finite difference to approximate the gradient. But how accurate is that approximation, and can we do better? This tutorial compares three numerical differentiation schemes:

- **Forward difference:** $f'(x) \approx \frac{f(x+\delta) - f(x)}{\delta}$ — $O(\delta)$ error, $n+1$ evaluations
- **Central difference:** $f'(x) \approx \frac{f(x+\delta) - f(x-\delta)}{2\delta}$ — $O(\delta^2)$ error, $2n$ evaluations
- **Complex step:** $f'(x) \approx \frac{\text{Im}[f(x + i\delta)]}{\delta}$ — machine-precision accuracy, but requires a complex-compatible function

The trade-off is always **accuracy vs. cost**.

## Key Takeaway

> **Central differences are the practical sweet spot: $O(\delta^2)$ accuracy at the cost of $2n$ function evaluations. The complex-step method achieves machine precision but requires the function to accept complex inputs.**

## Math Core

| Method | Formula | Truncation error | Evaluations (n vars) |
|--------|---------|-----------------|---------------------|
| Forward | $\frac{f(x+\delta) - f(x)}{\delta}$ | $O(\delta)$ | $n + 1$ |
| Central | $\frac{f(x+\delta) - f(x-\delta)}{2\delta}$ | $O(\delta^2)$ | $2n$ |
| Complex step | $\frac{\operatorname{Im}[f(x + i\delta)]}{\delta}$ | $O(\delta^2)$, no subtraction cancellation | $n$ |

For forward and central differences, there is an optimal $\delta$ that balances truncation error against floating-point round-off. Too small, and catastrophic cancellation destroys the answer.

## Code

We start with a simple scalar function where we know the exact derivative, then apply the ideas to our four-bar objective.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from dms.curves import CompareCurves
%matplotlib inline

# Constants (same as Tutorial 1)
L0, L1 = 1.0, 1.0
PX, PY = 0.5, 0.3
TARGETS = np.array([
    [1.0, 1.5], [0.75, 1.5], [0.5, 1.5],
    [0.25, 1.5], [0.0, 1.5], [-0.25, 1.5]
])
THETAS = np.linspace(np.deg2rad(60), np.deg2rad(120), len(TARGETS))

### Accuracy vs. step size on a known function

Let's use $f(x) = \sin(x)$ at $x = 1$, where the true derivative is $\cos(1)$.

In [ ]:
x0 = 1.0
exact = np.cos(x0)
deltas = np.logspace(-15, 0, 100)

err_fwd = np.abs([(np.sin(x0 + d) - np.sin(x0)) / d - exact
                   for d in deltas])
err_cen = np.abs([(np.sin(x0 + d) - np.sin(x0 - d)) / (2*d) - exact
                   for d in deltas])
err_cpx = np.abs([np.imag(np.sin(x0 + 1j*d)) / d - exact
                   for d in deltas])

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.loglog(deltas, err_fwd, 'b-', lw=2, label='Forward')
ax.loglog(deltas, err_cen, 'r-', lw=2, label='Central')
ax.loglog(deltas, err_cpx, 'g-', lw=2, label='Complex step')
ax.set_xlabel(r'Step size $\delta$')
ax.set_ylabel('Absolute error')
ax.set_title(r"Derivative accuracy: $f(x)=\sin(x)$ at $x=1$")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

Key observations from the log-log plot:

- **Forward difference** (blue): error decreases with slope 1 ($O(\delta)$) until round-off takes over near $\delta \approx 10^{-8}$.
- **Central difference** (red): error decreases with slope 2 ($O(\delta^2)$), bottoming out near $\delta \approx 10^{-5}$.
- **Complex step** (green): error keeps dropping — no subtractive cancellation! Achieves near machine precision for any tiny $\delta$.

### Implementing all three for a vector function

Now let's implement gradient functions that work with our 2-variable four-bar objective.

In [ ]:
def grad_forward(f, x, delta=1e-7):
    """Forward-difference gradient. Cost: n+1 evaluations."""
    x = np.array(x, dtype=float)
    fx = f(x)
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_fwd = x.copy()
        x_fwd[i] += delta
        grad[i] = (f(x_fwd) - fx) / delta
    return grad  # n+1 evaluations total


def grad_central(f, x, delta=1e-5):
    """Central-difference gradient. Cost: 2n evaluations."""
    x = np.array(x, dtype=float)
    grad = np.zeros_like(x)
    for i in range(len(x)):
        x_fwd, x_bwd = x.copy(), x.copy()
        x_fwd[i] += delta
        x_bwd[i] -= delta
        grad[i] = (f(x_fwd) - f(x_bwd)) / (2 * delta)
    return grad  # 2n evaluations total

In [ ]:
def fourbar_fk(theta1, l0, l1, l2, l3):
    """Cosine-law closed-form FK for four-bar linkage."""
    ax, ay = l1 * np.cos(theta1), l1 * np.sin(theta1)
    dx, dy = ax - l0, ay
    d = np.sqrt(dx**2 + dy**2)
    cos_alpha = (l3**2 + d**2 - l2**2) / (2 * l3 * d)
    if np.abs(cos_alpha) > 1:
        return None
    alpha = np.arccos(np.clip(cos_alpha, -1, 1))
    beta = np.arctan2(dy, dx)
    theta3 = beta + alpha
    bx = l0 + l3 * np.cos(theta3)
    by = l3 * np.sin(theta3)
    theta2 = np.arctan2(by - ay, bx - ax)
    return theta2, theta3


def coupler_point(theta1, l2, l3):
    """Compute coupler point for given crank angle."""
    sol = fourbar_fk(theta1, L0, L1, l2, l3)
    if sol is None:
        return None
    theta2, _ = sol
    ax, ay = L1 * np.cos(theta1), L1 * np.sin(theta1)
    ux, uy = np.cos(theta2), np.sin(theta2)
    return np.array([ax + PX * ux - PY * uy,
                     ay + PX * uy + PY * ux])


def objective(x):
    """Path-synthesis objective."""
    l2, l3 = x
    path = []
    for theta1 in THETAS:
        pt = coupler_point(theta1, l2, l3)
        if pt is None:
            return 1e3
        path.append(pt)
    path = np.array(path)
    return CompareCurves(path[:, 0], path[:, 1],
                         TARGETS[:, 0], TARGETS[:, 1])

### Comparing gradients on the four-bar problem

Let's evaluate both methods at a test point and see how they differ.

In [ ]:
x_test = np.array([1.5, 1.5])

g_fwd = grad_forward(objective, x_test, delta=1e-7)
g_cen = grad_central(objective, x_test, delta=1e-5)

print(f'Forward  gradient: [{g_fwd[0]:+.6f}, {g_fwd[1]:+.6f}]')
print(f'Central  gradient: [{g_cen[0]:+.6f}, {g_cen[1]:+.6f}]')
print(f'Difference:        [{abs(g_fwd[0]-g_cen[0]):.2e}, '
      f'{abs(g_fwd[1]-g_cen[1]):.2e}]')

### The cost of numerical gradients

With $n = 2$ design variables:
- Forward difference: $n + 1 = 3$ function evaluations per gradient
- Central difference: $2n = 4$ function evaluations per gradient

For gradient descent over $k$ iterations, that's $3k$ vs. $4k$ calls to `objective`. Central differences cost 33% more per step, but the improved accuracy often means fewer total steps to converge.

In [ ]:
def gradient_descent(f, x0, grad_fn, alpha=0.05, n_iter=150,
                     bounds=(0.3, 2.5)):
    """GD with a pluggable gradient function."""
    x = np.array(x0, dtype=float)
    history = [x.copy()]
    for _ in range(n_iter):
        g = grad_fn(f, x)
        x = x - alpha * g
        x = np.clip(x, bounds[0], bounds[1])
        history.append(x.copy())
    return np.array(history)


x0 = [1.8, 1.8]
h_fwd = gradient_descent(objective, x0, grad_forward)
h_cen = gradient_descent(objective, x0, grad_central)

c_fwd = [objective(h) for h in h_fwd]
c_cen = [objective(h) for h in h_cen]

fig, ax = plt.subplots(figsize=(7, 4))
ax.semilogy(c_fwd, 'b-', lw=2, label='Forward diff')
ax.semilogy(c_cen, 'r-', lw=2, label='Central diff')
ax.set_xlabel('Iteration')
ax.set_ylabel(r'$f(\mathbf{x})$')
ax.set_title('GD convergence: forward vs. central differences')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

### Sensitivity to step size $\delta$

Let's sweep $\delta$ and measure how the gradient estimate changes — analogous to the $\sin(x)$ experiment but on our real objective.

In [ ]:
deltas = np.logspace(-12, -1, 50)
# Use central diff with very small delta as reference
g_ref = grad_central(objective, x_test, delta=1e-5)

err_fwd_fb = [np.linalg.norm(grad_forward(objective, x_test, d) - g_ref)
              for d in deltas]
err_cen_fb = [np.linalg.norm(grad_central(objective, x_test, d) - g_ref)
              for d in deltas]

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(deltas, err_fwd_fb, 'b-', lw=2, label='Forward')
ax.loglog(deltas, err_cen_fb, 'r-', lw=2, label='Central')
ax.set_xlabel(r'Step size $\delta$')
ax.set_ylabel(r'$\|\nabla_{approx} - \nabla_{ref}\|$')
ax.set_title('Gradient error vs. step size (four-bar objective)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

## Mechanism Hook

Numerical gradients are general-purpose but expensive: every gradient costs $O(n)$ function evaluations. For the four-bar linkage, the FK equations are differentiable analytically — in Tutorial 13 we will derive **analytical gradients** that are both faster and exact. For now, central differences with $\delta \approx 10^{-5}$ are a solid default.

In [ ]:
from dms.mechanisms.fourbar import FourBar

x_best = h_cen[-1]
l2, l3 = x_best
mechanism = FourBar(L0, L1, l2, l3)
pts = [coupler_point(t, l2, l3) for t in THETAS]
path = np.array([p for p in pts if p is not None])

fig, ax = plt.subplots(figsize=(6, 5))
mechanism.plot(np.deg2rad(90), ax=ax)
ax.plot(TARGETS[:, 0], TARGETS[:, 1], 'r*', ms=10, label='targets')
if len(path) > 0:
    ax.plot(path[:, 0], path[:, 1], 'b.-', label='coupler path')
ax.set_aspect('equal')
ax.legend()
ax.set_title(f'Central-diff GD: $\\ell_2$={l2:.3f}, $\\ell_3$={l3:.3f}'
             f'  |  f = {objective(x_best):.4f}')
plt.tight_layout()

---